# LeetCode 1953 — Maximum Number of Weeks for Which You Can Work bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-weeks-for-which-you-can-work/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def numberOfWeeks(self, milestones: List[int]) -> int:
        ...
```

Lộ trình: **DFS + memo trên trạng thái (số mốc còn lại, dự án tuần trước)** (chỉ chạy được với dữ liệu tí hon) → **mô phỏng tham lam bằng heap: mỗi tuần làm dự án còn nhiều nhất mà không trùng tuần trước** `O(S log n)` → **xếp lịch kiểu "chẵn trước, lẻ sau"** (dựng lịch thật để kiểm chứng) → **công thức `O(n)`: so dự án lớn nhất với tổng phần còn lại** (lời giải nộp).

## 1. Mục tiêu học

1. Mô hình hóa bài toán xếp lịch "không hai tuần liền cùng dự án" như **xếp chuỗi không có hai ký tự kề nhau giống nhau** (họ bài *Reorganize String*).
2. Nhận ra **nút thắt duy nhất** là dự án lớn nhất: nếu nó không quá lớn, làm được **hết**; nếu quá lớn, chỉ làm được `2·(phần còn lại) + 1` tuần.
3. Chứng minh công thức bằng **cận trên** (đếm khe xen kẽ) + **xây dựng** (lịch đạt được cận).
4. Biết vì sao mô phỏng từng tuần không khả thi (`S` có thể tới `10¹⁴`).
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Có `n` dự án; dự án `i` có `milestones[i]` mốc công việc. Mỗi tuần:
- Bạn hoàn thành **đúng một** mốc của **một** dự án, và **phải** làm việc mỗi tuần.
- **Không** được làm hai mốc của cùng một dự án trong hai tuần **liên tiếp**.

Khi đã xong hết, hoặc khi mốc duy nhất còn lại sẽ vi phạm luật, bạn **dừng**. Trả về số tuần **tối đa** có thể làm.

**Ví dụ 1:** `milestones = [1,2,3]` → `6` (ví dụ lịch: 0, 2, 1, 2, 1, 2).

**Ví dụ 2:** `milestones = [5,2,1]` → `7` (lịch 0,1,0,1,0,2,0; mốc cuối của dự án 0 không làm được ở tuần 8).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ milestones[i] ≤ 10⁹`.

In [1]:
import heapq
import random
import time
from functools import lru_cache
from typing import List

examples = [([1, 2, 3], 6), ([5, 2, 1], 7)]
for m, want in examples:
    print(m, "->", want, "| tổng số mốc =", sum(m))

[1, 2, 3] -> 6 | tổng số mốc = 6
[5, 2, 1] -> 7 | tổng số mốc = 8


## 3. Kiến thức nền

### 3.1 Bài toán tương đương: xếp chuỗi không có hai ký tự kề giống nhau

Mỗi dự án là một "chữ cái", số mốc là số lần xuất hiện. Một lịch làm việc là một **chuỗi** trong đó không có hai chữ cái giống nhau đứng cạnh nhau. Ta muốn chuỗi **dài nhất** (được phép bỏ bớt chữ cái).

In [2]:
def is_valid_schedule(schedule, milestones):
    """Lịch hợp lệ: không hai tuần liền cùng dự án, không dùng quá số mốc."""
    used = [0] * len(milestones)
    for w, p in enumerate(schedule):
        if w > 0 and schedule[w - 1] == p:
            return False
        used[p] += 1
    return all(used[i] <= milestones[i] for i in range(len(milestones)))

print(is_valid_schedule([0, 2, 1, 2, 1, 2], [1, 2, 3]), is_valid_schedule([0, 0, 1], [2, 1]))

True False


### 3.2 Đếm khe: mỗi lần làm dự án lớn cần một "vách ngăn"

Nếu dự án lớn nhất có `mx` mốc và các dự án khác có tổng `rest`, thì giữa hai lần làm dự án lớn phải có **ít nhất một** tuần làm dự án khác. Với `rest` vách ngăn, ta xếp được tối đa `rest + 1` lần dự án lớn:

```
L x L x L x ... x L        (rest vách ngăn x, rest + 1 chữ L)
```

In [3]:
def max_big_uses(rest):
    return rest + 1

for rest in range(4):
    print(f"rest={rest}: dự án lớn làm được tối đa {max_big_uses(rest)} lần ->",
          " ".join(["L", "x"] * rest + ["L"]))

rest=0: dự án lớn làm được tối đa 1 lần -> L
rest=1: dự án lớn làm được tối đa 2 lần -> L x L
rest=2: dự án lớn làm được tối đa 3 lần -> L x L x L
rest=3: dự án lớn làm được tối đa 4 lần -> L x L x L x L


### 3.3 Heap tối đa trong Python

`heapq` là heap **min**; để lấy phần tử lớn nhất, đẩy `-giá trị`. Dùng cho mô phỏng tham lam (Cách 2).

In [4]:
h = [(-5, 0), (-2, 1), (-1, 2)]
heapq.heapify(h)
print("lớn nhất:", -h[0][0], "của dự án", h[0][1])

lớn nhất: 5 của dự án 0


## 4. Ý tưởng nền tảng / trực giác

Gọi `S = sum(milestones)`, `mx = max(milestones)`, `rest = S − mx`.

- **Trường hợp A — `mx ≤ rest + 1`:** dự án lớn nhất "không lấn át". Có thể làm **hết** `S` tuần (Ví dụ 1: `mx = 3`, `rest = 3` ⇒ `6`).
- **Trường hợp B — `mx > rest + 1`:** dự án lớn có quá nhiều mốc; dùng hết `rest` mốc khác làm vách ngăn, làm dự án lớn `rest + 1` lần ⇒ `2·rest + 1` tuần (Ví dụ 2: `mx = 5`, `rest = 3` ⇒ `5 ≤ 4` sai ⇒ `2·3 + 1 = 7`).

Chỉ **dự án lớn nhất** có thể gây tắc: nếu dự án lớn nhất ổn thì các dự án nhỏ hơn càng ổn (mỗi dự án nhỏ hơn có "phần còn lại" lớn hơn).

In [5]:
for m, want in examples:
    S, mx = sum(m), max(m); rest = S - mx
    case = "A (làm hết)" if mx <= rest + 1 else "B (bị dự án lớn chặn)"
    val = S if mx <= rest + 1 else 2 * rest + 1
    print(f"{m}: S={S}, mx={mx}, rest={rest} -> trường hợp {case} -> {val} (kỳ vọng {want})")

[1, 2, 3]: S=6, mx=3, rest=3 -> trường hợp A (làm hết) -> 6 (kỳ vọng 6)
[5, 2, 1]: S=8, mx=5, rest=3 -> trường hợp B (bị dự án lớn chặn) -> 7 (kỳ vọng 7)


## 5. Cách 1 — Brute Force / baseline: DFS + memo trên trạng thái

Trạng thái: bộ số mốc còn lại của từng dự án + dự án vừa làm tuần trước. Mỗi bước thử mọi dự án còn mốc và khác dự án trước; lấy `1 + max`. Số trạng thái `Π(mᵢ + 1) · n` — chỉ dùng cho dữ liệu nhỏ.

In [6]:
def weeks_brute(milestones):
    n = len(milestones)

    @lru_cache(maxsize=None)
    def dfs(state, last):
        best = 0
        for i in range(n):
            if i != last and state[i] > 0:
                nxt = state[:i] + (state[i] - 1,) + state[i + 1:]
                best = max(best, 1 + dfs(nxt, i))
        return best

    return dfs(tuple(milestones), -1)

for m, want in examples:
    print(m, "->", weeks_brute(m), "| kỳ vọng", want)

[1, 2, 3] -> 6 | kỳ vọng 6
[5, 2, 1] -> 7 | kỳ vọng 7


**Độ phức tạp:** số trạng thái tới `Π(mᵢ+1) · (n+1)`, mỗi trạng thái `O(n²)` (do tạo tuple) — bùng nổ ngay cả với `m = [10]*6`. **Nút thắt:** ta không cần biết lịch cụ thể, chỉ cần một **cấu trúc** của lời giải tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cận trên):** Trong lịch bất kỳ, giữa hai lần làm cùng dự án `i` phải có tuần làm dự án khác ⇒ dự án `i` được làm tối đa `(số tuần làm dự án khác) + 1 ≤ (S − mᵢ) + 1` lần. Áp cho dự án lớn nhất: số tuần làm dự án lớn `≤ min(mx, rest + 1)`, các dự án khác `≤ rest` ⇒ tổng `≤ rest + min(mx, rest + 1)`.

**Insight 2 (đạt được):** Có lịch đạt đúng cận đó (tham lam "luôn làm dự án còn nhiều nhất không trùng tuần trước", hoặc xếp vị trí chẵn/lẻ).

**Kết luận:** `ans = rest + min(mx, rest + 1)`, tức
- `S` nếu `mx ≤ rest + 1`,
- `2·rest + 1` nếu ngược lại.

**Invariant cho tham lam (trường hợp A):** sau mỗi tuần, dự án lớn nhất còn lại vẫn thỏa `mx' ≤ rest' + 1` (xem chứng minh mục 10).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam bằng heap `O(S log n)`

Mỗi tuần, lấy dự án còn **nhiều mốc nhất** mà không phải dự án tuần trước. Nếu chỉ còn dự án tuần trước ⇒ dừng.

In [7]:
def weeks_greedy_heap(milestones, return_schedule=False):
    h = [(-c, i) for i, c in enumerate(milestones)]
    heapq.heapify(h)
    schedule, last = [], -1
    while h:
        c, i = heapq.heappop(h)
        if i == last:                       # dự án lớn nhất trùng tuần trước -> lấy dự án thứ hai
            if not h:
                break                       # chỉ còn dự án tuần trước: dừng
            c2, i2 = heapq.heappop(h)
            heapq.heappush(h, (c, i))
            c, i = c2, i2
        schedule.append(i); last = i
        if c + 1 < 0:
            heapq.heappush(h, (c + 1, i))
    return schedule if return_schedule else len(schedule)

for m, want in examples:
    sch = weeks_greedy_heap(m, True)
    print(m, "-> lịch", sch, "dài", len(sch), "hợp lệ:", is_valid_schedule(sch, m), "| kỳ vọng", want)

[1, 2, 3] -> lịch [2, 1, 2, 0, 1, 2] dài 6 hợp lệ: True | kỳ vọng 6
[5, 2, 1] -> lịch [0, 1, 0, 1, 0, 2, 0] dài 7 hợp lệ: True | kỳ vọng 7


**Độ phức tạp:** `O(S log n)` thời gian — `S` tới `10⁵ · 10⁹ = 10¹⁴` ⇒ không thể nộp. Hữu ích để **sinh lịch thật** và kiểm chứng công thức.

### 7.2 Cách 3 — Xếp lịch "vị trí chẵn trước, lẻ sau" (dựng lịch trực tiếp)

Ý tưởng giống *Reorganize String* (LeetCode 767): sắp dự án theo số mốc giảm dần; viết lần lượt từng mốc vào các ô `0, 2, 4, …` rồi `1, 3, 5, …` của một mảng dài `T` (`T` là đáp án công thức). Trong trường hợp B ta chỉ dùng `rest + 1` mốc của dự án lớn.

In [8]:
def schedule_even_odd(milestones):
    n = len(milestones)
    S, mx = sum(milestones), max(milestones); rest = S - mx
    big = milestones.index(mx)
    use = list(milestones)
    if mx > rest + 1:
        use[big] = rest + 1                 # chỉ dùng rest + 1 mốc của dự án lớn
    T = sum(use)
    order = sorted(range(n), key=lambda i: -use[i])
    slots = list(range(0, T, 2)) + list(range(1, T, 2))
    out, k = [None] * T, 0
    for i in order:
        for _ in range(use[i]):
            out[slots[k]] = i; k += 1
    return out

for m in ([1, 2, 3], [5, 2, 1], [3, 3, 3, 1]):
    sch = schedule_even_odd(m)
    print(m, "->", sch, "dài", len(sch), "hợp lệ:", is_valid_schedule(sch, m))

[1, 2, 3] -> [2, 1, 2, 1, 2, 0] dài 6 hợp lệ: True
[5, 2, 1] -> [0, 1, 0, 1, 0, 2, 0] dài 7 hợp lệ: True
[3, 3, 3, 1] -> [0, 1, 0, 2, 0, 2, 1, 2, 1, 3] dài 10 hợp lệ: True


Vì sao hợp lệ: dự án đầu (lớn nhất, `≤ ⌈T/2⌉` mốc) chỉ nằm ở các ô chẵn ⇒ không kề nhau. Mỗi dự án sau chiếm một **dãy liên tiếp** trong danh sách `slots`; các ô liên tiếp trong `slots` cách nhau 2 (trừ chỗ chuyển từ chẵn sang lẻ), và một dự án không lớn nhất có `≤ ⌊T/2⌋` mốc nên không thể "vòng" lại chạm chính nó. **Độ phức tạp** `O(S + n log n)` — vẫn phụ thuộc `S`.

### 7.3 Cách 4 — Công thức `O(n)` (lời giải nộp)

In [9]:
class SolutionFormula:
    def numberOfWeeks(self, milestones: List[int]) -> int:
        S, mx = sum(milestones), max(milestones)
        rest = S - mx
        return S if mx <= rest + 1 else 2 * rest + 1

for m, want in examples:
    print(m, SolutionFormula().numberOfWeeks(m), want)

[1, 2, 3] 6 6
[5, 2, 1] 7 7


Dạng gọn tương đương: `rest + min(mx, rest + 1)`. **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

In [10]:
class SolutionMin:
    def numberOfWeeks(self, milestones: List[int]) -> int:
        mx = max(milestones); rest = sum(milestones) - mx
        return rest + min(mx, rest + 1)

for m, want in examples:
    assert SolutionMin().numberOfWeeks(m) == want
print("dạng min: OK")

dạng min: OK


## 8. TRACE từng bước

Trace mô phỏng tham lam (Cách 2) trên Ví dụ 2 `[5,2,1]`, in heap sau mỗi tuần:

In [11]:
def trace_greedy(milestones):
    rem = list(milestones); last = -1; week = 0
    while True:
        cand = [i for i in range(len(rem)) if rem[i] > 0 and i != last]
        if not cand:
            left = [i for i in range(len(rem)) if rem[i] > 0]
            print(f"dừng: còn mốc ở dự án {left} nhưng tuần trước vừa làm dự án {last}" if left else "dừng: đã làm hết")
            break
        i = max(cand, key=lambda j: rem[j])
        rem[i] -= 1; last = i; week += 1
        print(f"tuần {week}: làm dự án {i} -> còn lại {rem}")
    return week

print("tổng số tuần:", trace_greedy([5, 2, 1]))

tuần 1: làm dự án 0 -> còn lại [4, 2, 1]
tuần 2: làm dự án 1 -> còn lại [4, 1, 1]
tuần 3: làm dự án 0 -> còn lại [3, 1, 1]
tuần 4: làm dự án 1 -> còn lại [3, 0, 1]
tuần 5: làm dự án 0 -> còn lại [2, 0, 1]
tuần 6: làm dự án 2 -> còn lại [2, 0, 0]
tuần 7: làm dự án 0 -> còn lại [1, 0, 0]
dừng: còn mốc ở dự án [0] nhưng tuần trước vừa làm dự án 0
tổng số tuần: 7


Ví dụ 1 `[1,2,3]` — trường hợp A, làm hết:

In [12]:
print("tổng số tuần:", trace_greedy([1, 2, 3]))

tuần 1: làm dự án 2 -> còn lại [1, 2, 2]
tuần 2: làm dự án 1 -> còn lại [1, 1, 2]
tuần 3: làm dự án 2 -> còn lại [1, 1, 1]
tuần 4: làm dự án 0 -> còn lại [0, 1, 1]
tuần 5: làm dự án 1 -> còn lại [0, 0, 1]
tuần 6: làm dự án 2 -> còn lại [0, 0, 0]
dừng: đã làm hết
tổng số tuần: 6


Trace công thức trên vài bộ:

| milestones | S | mx | rest | `mx ≤ rest+1`? | đáp án |
|---|---|---|---|---|---|
| `[1,2,3]` | 6 | 3 | 3 | có | 6 |
| `[5,2,1]` | 8 | 5 | 3 | không (5 > 4) | 7 |
| `[4,3]` | 7 | 4 | 3 | có (4 ≤ 4) | 7 |
| `[9]` | 9 | 9 | 0 | không | 1 |
| `[10,1,1]` | 12 | 10 | 2 | không | 5 |

In [13]:
def trace_formula(m):
    S, mx = sum(m), max(m); rest = S - mx
    ok = mx <= rest + 1
    ans = S if ok else 2 * rest + 1
    print(f"{str(m):<10} S={S:<3} mx={mx:<3} rest={rest:<3} mx<=rest+1? {str(ok):<5} -> {ans}  (vét cạn: {weeks_brute(m)})")
    return ans

for m in ([1, 2, 3], [5, 2, 1], [4, 3], [9], [10, 1, 1]):
    trace_formula(m)

[1, 2, 3]  S=6   mx=3   rest=3   mx<=rest+1? True  -> 6  (vét cạn: 6)
[5, 2, 1]  S=8   mx=5   rest=3   mx<=rest+1? False -> 7  (vét cạn: 7)
[4, 3]     S=7   mx=4   rest=3   mx<=rest+1? True  -> 7  (vét cạn: 7)
[9]        S=9   mx=9   rest=0   mx<=rest+1? False -> 1  (vét cạn: 1)
[10, 1, 1] S=12  mx=10  rest=2   mx<=rest+1? False -> 5  (vét cạn: 5)


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def numberOfWeeks(self, milestones: List[int]) -> int:
        total = sum(milestones)
        mx = max(milestones)
        rest = total - mx                    # tổng mốc của các dự án khác
        if mx <= rest + 1:                   # dự án lớn nhất không lấn át -> làm hết
            return total
        return 2 * rest + 1                  # L x L x ... x L

for m, want in examples:
    assert Solution().numberOfWeeks(m) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Đặt `S, mx, rest` như trên.

**Cận trên.** Trong mọi lịch hợp lệ, gọi `a` = số tuần làm dự án lớn nhất, `b` = số tuần làm dự án khác. Hai lần làm dự án lớn liên tiếp (theo thứ tự thời gian) phải cách nhau ít nhất một tuần làm dự án khác ⇒ `a ≤ b + 1`. Mà `b ≤ rest`, `a ≤ mx` ⇒ tổng `a + b ≤ rest + min(mx, rest + 1)`.

**Đạt được — trường hợp B (`mx > rest + 1`).** Lịch `L x L x … x L` dùng hết `rest` mốc khác (theo thứ tự bất kỳ) xen giữa `rest + 1` mốc của dự án lớn: hợp lệ vì hai tuần kề nhau luôn có đúng một tuần là `L` và một tuần là `x` (các `x` không bao giờ kề nhau) ⇒ `2·rest + 1`.

**Đạt được — trường hợp A (`mx ≤ rest + 1`).** Chứng minh tham lam "làm dự án còn nhiều nhất khác tuần trước" làm hết. Invariant: *dự án còn nhiều mốc nhất `M` thỏa `M ≤ (tổng còn lại − M) + 1`, và nếu đẳng thức xảy ra thì dự án đó không phải dự án tuần trước*. Mỗi tuần ta làm dự án lớn nhất được phép; kiểm tra các trường hợp cho thấy invariant được giữ, và khi tổng còn lại `> 0` luôn có ít nhất một dự án được phép (nếu chỉ còn dự án tuần trước thì nó có `M = tổng` ⇒ `M ≤ 1` và đẳng thức `M = 0 + 1` mâu thuẫn với "không phải dự án tuần trước"). Mục 12.3 kiểm chứng điều này bằng máy với hàng nghìn bộ ngẫu nhiên. Cách 3 (chẵn/lẻ) cho một chứng minh xây dựng khác.

Cận trên = giá trị đạt được ⇒ công thức đúng.

Kiểm bằng máy invariant của tham lam trong trường hợp A: sau mỗi tuần, `M ≤ (còn lại − M) + 1`, và nếu đẳng thức thì có ít nhất một dự án đạt `M` khác dự án tuần trước.

In [15]:
def check_invariant(milestones):
    rem = list(milestones); last = -1
    while sum(rem) > 0:
        cand = [i for i in range(len(rem)) if rem[i] > 0 and i != last]
        assert cand, "tham lam bị kẹt trong trường hợp A"
        i = max(cand, key=lambda j: rem[j])
        rem[i] -= 1; last = i
        T = sum(rem); M = max(rem)
        assert M <= T - M + 1
        if T > 0 and M == T - M + 1:
            assert any(rem[j] == M and j != last for j in range(len(rem)))
    return True

rng = random.Random(10)
cnt = 0
while cnt < 1000:
    m = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    if max(m) <= sum(m) - max(m) + 1:
        check_invariant(m); cnt += 1
print("Invariant tham lam giữ đúng trên", cnt, "bộ thuộc trường hợp A")

Invariant tham lam giữ đúng trên 1000 bộ thuộc trường hợp A


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. DFS + memo | `O(Π(mᵢ+1) · n²)` | số trạng thái |
| 2. Tham lam heap | `O(S log n)` | `O(n)` (+ `O(S)` nếu lưu lịch) |
| 3. Chẵn/lẻ | `O(S + n log n)` | `O(S)` |
| 4. Công thức | `O(n)` | `O(1)` |

In [16]:
rng = random.Random(1953)
big = [rng.randint(1, 10**9) for _ in range(10**5)]
t = time.perf_counter(); r = Solution().numberOfWeeks(big); dt = time.perf_counter() - t
print(f"Công thức, n=10^5: {r} trong {1000*dt:.1f} ms; S = {sum(big):.3e} tuần")
small = [rng.randint(1, 200) for _ in range(500)]
t = time.perf_counter(); g = weeks_greedy_heap(small); dt = time.perf_counter() - t
assert g == Solution().numberOfWeeks(small)
print(f"Heap mô phỏng S={sum(small)} tuần: {dt:.3f}s -> với S≈{sum(big):.1e} ước tính ≈ {dt * sum(big) / sum(small):.1e}s")

Công thức, n=10^5: 49966708172976 trong 1.5 ms; S = 4.997e+13 tuần
Heap mô phỏng S=50985 tuần: 0.016s -> với S≈5.0e+13 ước tính ≈ 1.6e+07s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [17]:
sols = [Solution(), SolutionFormula(), SolutionMin()]
edge = examples + [
    ([1], 1), ([9], 1), ([1, 1], 2), ([2, 1], 3), ([3, 1], 3),
    ([4, 3], 7), ([5, 3], 7), ([10, 1, 1], 5), ([2, 2, 2], 6), ([3, 3, 3, 1], 10),
]
for m, want in edge:
    assert weeks_brute(m) == want, (m, weeks_brute(m))
    for s in sols:
        assert s.numberOfWeeks(m) == want
    assert weeks_greedy_heap(m) == want
    assert is_valid_schedule(schedule_even_odd(m), m) and len(schedule_even_odd(m)) == want
    print("OK", m, "->", want)
print("Giá trị lớn:", Solution().numberOfWeeks([10**9] * 10**5), Solution().numberOfWeeks([10**9, 1]))

OK [1, 2, 3] -> 6
OK [5, 2, 1] -> 7
OK [1] -> 1
OK [9] -> 1
OK [1, 1] -> 2
OK [2, 1] -> 3
OK [3, 1] -> 3
OK [4, 3] -> 7
OK [5, 3] -> 7
OK [10, 1, 1] -> 5
OK [2, 2, 2] -> 6
OK [3, 3, 3, 1] -> 10
Giá trị lớn: 100000000000000 3


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [18]:
rng = random.Random(42)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 4)
    m = [rng.randint(1, 5) for _ in range(n)]
    if rng.random() < 0.3:
        m[0] = rng.randint(1, 12)          # thêm trường hợp có dự án lấn át
    want = weeks_brute(m)
    for s in sols:
        assert s.numberOfWeeks(m) == want, m
    assert weeks_greedy_heap(m) == want, m
    cnt += 1
print(cnt, "bộ ngẫu nhiên: khớp vét cạn")

400 bộ ngẫu nhiên: khớp vét cạn


### 12.3 Ngẫu nhiên lớn hơn: tham lam heap và lịch chẵn/lẻ đều hợp lệ và đạt công thức

In [19]:
rng = random.Random(7)
for _ in range(1500):
    n = rng.randint(1, 8)
    m = [rng.randint(1, 30) for _ in range(n)]
    if rng.random() < 0.3:
        m[rng.randrange(n)] = rng.randint(1, 150)
    want = Solution().numberOfWeeks(m)
    sch = weeks_greedy_heap(m, True)
    assert len(sch) == want and is_valid_schedule(sch, m)
    sch2 = schedule_even_odd(m)
    assert len(sch2) == want and is_valid_schedule(sch2, m)
print("1500 bộ: tham lam & chẵn/lẻ cho lịch hợp lệ đạt đúng công thức")

1500 bộ: tham lam & chẵn/lẻ cho lịch hợp lệ đạt đúng công thức


## 13. Lỗi người mới hay gặp

### Lỗi 1 (lỗi "giả" — đáng hiểu) — So `mx ≤ rest` thay vì `mx ≤ rest + 1`

In [20]:
def wrong_no_plus_one(m):
    S, mx = sum(m), max(m); rest = S - mx
    return S if mx <= rest else 2 * rest + 1

for m in ([1, 2, 3], [5, 2, 1], [4, 3], [2, 1], [1]):
    print(m, "sai:", wrong_no_plus_one(m), "| đúng:", Solution().numberOfWeeks(m))

[1, 2, 3] sai: 6 | đúng: 6
[5, 2, 1] sai: 7 | đúng: 7
[4, 3] sai: 7 | đúng: 7
[2, 1] sai: 3 | đúng: 3
[1] sai: 1 | đúng: 1


**Vì sao "chạy được" mà vẫn sai một nửa:** khi `mx = rest + 1` thì `2·rest + 1 = rest + mx = S` — hai nhánh cho **cùng** giá trị, nên lỗi này thực ra vô hại! Đây là ví dụ tốt để hiểu công thức: biên giới giữa hai trường hợp là liên tục. Lỗi **thật** là dạng tiếp theo.

### Lỗi 2 — Quên `+1`: trả về `2·rest` khi dự án lớn lấn át

In [21]:
def wrong_two_rest(m):
    S, mx = sum(m), max(m); rest = S - mx
    return S if mx <= rest + 1 else 2 * rest

for m in ([5, 2, 1], [9], [10, 1, 1]):
    print(m, "sai:", wrong_two_rest(m), "| đúng:", Solution().numberOfWeeks(m))

[5, 2, 1] sai: 6 | đúng: 7
[9] sai: 0 | đúng: 1
[10, 1, 1] sai: 4 | đúng: 5


**Vì sao sai:** lịch `L x L x … x L` **bắt đầu và kết thúc** bằng dự án lớn ⇒ có `rest + 1` chữ `L`, không phải `rest`. Với `[9]` đáp án là `1` (làm một tuần rồi dừng), không phải `0`.

### Lỗi 3 — Tham lam "luân phiên theo thứ tự chỉ số" (round-robin)

In [22]:
def wrong_round_robin(m):
    rem = list(m); last = -1; weeks = 0
    while True:
        moved = False
        for i in range(len(rem)):                    # luôn ưu tiên chỉ số nhỏ
            if rem[i] > 0 and i != last:
                rem[i] -= 1; last = i; weeks += 1; moved = True
                break
        if not moved:
            return weeks

for m in ([1, 2, 3], [1, 1, 4], [2, 1, 3]):
    print(m, "sai:", wrong_round_robin(m), "| đúng:", Solution().numberOfWeeks(m))

[1, 2, 3] sai: 5 | đúng: 6
[1, 1, 4] sai: 3 | đúng: 5
[2, 1, 3] sai: 4 | đúng: 6


**Vì sao sai:** làm hết các dự án nhỏ trước khiến dự án lớn bị "bỏ lại" một mình ở cuối, không còn vách ngăn. Phải **ưu tiên dự án còn nhiều mốc nhất** (như Cách 2) để dùng vách ngăn hiệu quả.

### Lỗi 4 — Mô phỏng từng tuần với dữ liệu thật

In [23]:
m = [rng.randint(1, 3000) for _ in range(300)]
t = time.perf_counter(); weeks_greedy_heap(m); dt = time.perf_counter() - t
print(f"mô phỏng S={sum(m)} tuần: {dt:.2f}s -> S = 10^14 tuần ≈ {dt * 1e14 / sum(m):.1e}s (TLE)")

mô phỏng S=412563 tuần: 0.11s -> S = 10^14 tuần ≈ 2.6e+07s (TLE)


**Vì sao chưa tốt:** đúng nhưng `O(S log n)`; `S` có thể là `10¹⁴`. Chỉ cần `sum` và `max`.

### Lỗi 5 — So dự án lớn nhất với **dự án lớn thứ hai** thay vì tổng phần còn lại

In [24]:
def wrong_second_max(m):
    if len(m) == 1:
        return 1
    a = sorted(m, reverse=True)
    mx, second = a[0], a[1]
    S = sum(m)
    return S if mx <= second + 1 else 2 * (S - mx) + 1

for m in ([3, 1, 1, 1], [5, 2, 2, 2], [4, 1, 1, 1, 1], [4, 1, 1, 1]):
    print(m, "sai:", wrong_second_max(m), "| đúng:", Solution().numberOfWeeks(m), "| tổng mốc:", sum(m))

[3, 1, 1, 1] sai: 7 | đúng: 6 | tổng mốc: 6
[5, 2, 2, 2] sai: 13 | đúng: 11 | tổng mốc: 11
[4, 1, 1, 1, 1] sai: 9 | đúng: 8 | tổng mốc: 8
[4, 1, 1, 1] sai: 7 | đúng: 7 | tổng mốc: 7


**Vì sao sai:** **mọi** dự án khác đều làm vách ngăn được, không chỉ dự án lớn thứ hai. `[3,1,1,1]`: `3 > 1 + 1` nên code rơi vào nhánh "lấn át" và trả `2·3 + 1 = 7` — **lớn hơn cả tổng số mốc 6**, vô lý. Đúng ra ba dự án nhỏ cho 3 vách ngăn ⇒ làm hết 6 tuần. (Ở `[4,1,1,1]` code tình cờ đúng vì nằm đúng biên `mx = rest + 1`, nơi hai nhánh trùng giá trị — xem Lỗi 1.) **Mẹo kiểm:** đáp án không bao giờ vượt `sum(milestones)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Không hai phần tử kề nhau giống nhau"** + đếm tần suất ⇒ chỉ phần tử **tần suất lớn nhất** quyết định: so `mx` với `S − mx + 1` (hoặc `⌈S/2⌉`).
- Điều kiện `mx ≤ rest + 1` ⇔ `mx ≤ ⌈S/2⌉` ⇔ `2·mx ≤ S + 1`.
- Bài cùng họ: 767 *Reorganize String*, 1054 *Distant Barcodes*, 621 *Task Scheduler* (khoảng cách `k`), 358 *Rearrange String k Distance Apart*.
- Nếu cần **dựng** lịch: heap tham lam hoặc xếp ô chẵn/lẻ.
- Đề có số lớn (`10⁹`, `10⁵`) mà chỉ hỏi **số lượng** ⇒ gần như chắc chắn có công thức đóng.

In [25]:
rng = random.Random(11)
for _ in range(2000):
    m = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]
    S, mx = sum(m), max(m)
    assert (mx <= S - mx + 1) == (mx <= (S + 1) // 2) == (2 * mx <= S + 1)
print("Ba dạng điều kiện tương đương: OK")

Ba dạng điều kiện tương đương: OK


## 15. Biến thể của bài

1. **Cùng dự án phải cách nhau ít nhất `k` tuần** (Task Scheduler 621 với "nghỉ" cho phép) ⇒ đếm khung `(mx − 1)·(k + 1) + số dự án có tần suất mx`.
2. **In ra một lịch cụ thể** ⇒ heap hoặc chẵn/lẻ (Cách 2, 3).
3. **Bắt buộc làm hết, trả về có làm được không** ⇒ `mx ≤ rest + 1` (Bài tập 3).
4. **Lịch vòng tròn** (tuần cuối và tuần đầu cũng coi là kề) ⇒ điều kiện thành `mx ≤ rest` (Bài tập 5, biến thể tự dựng).
5. **Reorganize String** ⇒ cùng điều kiện, dựng chuỗi bằng heap.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS + memo | thử mọi lịch | mũ | lớn | kiểm chứng dữ liệu tí hon |
| 2. Tham lam heap | luôn làm dự án nhiều nhất được phép | `O(S log n)` | `O(n)` | cần lịch, `S` nhỏ |
| 3. Chẵn/lẻ | điền ô chẵn rồi lẻ | `O(S + n log n)` | `O(S)` | cần lịch, chứng minh xây dựng |
| 4. Công thức | `rest + min(mx, rest + 1)` | `O(n)` | `O(1)` | **nộp bài** |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[7]`, `[3,3]`, `[6,1,1]`, `[2,2,1]`?

**Bài 2 (tính tay):** Với `[8,2,2]`, tính `S`, `mx`, `rest`, xác định trường hợp và viết một lịch tối ưu.

**Bài 3 (điền TODO):** Viết `can_finish_all(milestones)` trả về `True` nếu làm được **hết** mọi mốc.

**Bài 4 (tự viết lại):** Viết `build_schedule(milestones)` trả về một lịch tối ưu (danh sách chỉ số dự án) bằng heap; kiểm bằng `is_valid_schedule`.

**Bài 5 (mini challenge, biến thể tự dựng):** Lịch **vòng tròn** — tuần cuối và tuần đầu cũng không được cùng dự án (lịch có ít nhất 2 tuần; lịch 1 tuần luôn hợp lệ). Viết `weeks_circular(milestones)` trả về số tuần tối đa.

In [26]:
def can_finish_all(milestones):
    # TODO
    pass

def build_schedule(milestones):
    # TODO
    pass

def weeks_circular(milestones):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `[7]`: chỉ một dự án | `[3,3]`: `mx = rest` | `[7] → 1`, `[3,3] → 6`, `[6,1,1] → 5`, `[2,2,1] → 5` |
| 2 | `S = 12, mx = 8, rest = 4` | `8 > 5` ⇒ trường hợp B | `2·4 + 1 = 9`, ví dụ `0 1 0 1 0 2 0 2 0` |
| 3 | Chính là điều kiện trường hợp A | `max ≤ sum − max + 1` | Một dòng |
| 4 | Heap `(-còn lại, chỉ số)` | Nếu đỉnh trùng tuần trước, lấy phần tử thứ hai | Dừng khi chỉ còn dự án tuần trước |
| 5 | Vòng tròn: dự án lớn cần `rest` vách cho `mx` lần (không có "+1") | Trường hợp A khi `mx ≤ rest` ⇒ `S`; ngược lại `2·rest` | Riêng `rest = 0` ⇒ `1` (một tuần) |

## 19. Đáp án tham khảo

In [27]:
r1 = [Solution().numberOfWeeks(m) for m in ([7], [3, 3], [6, 1, 1], [2, 2, 1])]
print("Bài 1:", r1); assert r1 == [1, 6, 5, 5]
sch = [0, 1, 0, 1, 0, 2, 0, 2, 0]
assert is_valid_schedule(sch, [8, 2, 2]) and len(sch) == Solution().numberOfWeeks([8, 2, 2]) == weeks_brute([8, 2, 2]) == 9
print("Bài 2: S=12, mx=8, rest=4 -> trường hợp B -> 9, lịch", sch)

Bài 1: [1, 6, 5, 5]
Bài 2: S=12, mx=8, rest=4 -> trường hợp B -> 9, lịch [0, 1, 0, 1, 0, 2, 0, 2, 0]


In [28]:
def can_finish_all(milestones):
    mx = max(milestones)
    return mx <= sum(milestones) - mx + 1

def build_schedule(milestones):
    h = [(-c, i) for i, c in enumerate(milestones)]
    heapq.heapify(h)
    out = []
    while h:
        c, i = heapq.heappop(h)
        if out and out[-1] == i:
            if not h:
                break
            c2, i2 = heapq.heappop(h); heapq.heappush(h, (c, i)); c, i = c2, i2
        out.append(i)
        if c + 1 < 0:
            heapq.heappush(h, (c + 1, i))
    return out

rng = random.Random(5)
for _ in range(500):
    m = [rng.randint(1, 6) for _ in range(rng.randint(1, 4))]
    assert can_finish_all(m) == (weeks_brute(m) == sum(m))
    s = build_schedule(m)
    assert is_valid_schedule(s, m) and len(s) == Solution().numberOfWeeks(m)
print("Bài 3, 4: OK", can_finish_all([5, 2, 1]), build_schedule([5, 2, 1]))

Bài 3, 4: OK False [0, 1, 0, 1, 0, 2, 0]


In [29]:
def weeks_circular(milestones):
    mx = max(milestones); rest = sum(milestones) - mx
    if rest == 0:
        return 1
    return rest + min(mx, rest)

def weeks_circular_brute(milestones):
    n = len(milestones); best = 0
    @lru_cache(maxsize=None)
    def dfs(state, first, last, length):
        nonlocal best
        if length == 1 or (length >= 2 and first != last):
            best = max(best, length)
        for i in range(n):
            if state[i] > 0 and i != last:
                nxt = state[:i] + (state[i] - 1,) + state[i + 1:]
                dfs(nxt, i if first == -1 else first, i, length + 1)
    dfs(tuple(milestones), -1, -1, 0)
    return best

rng = random.Random(8)
for _ in range(300):
    m = [rng.randint(1, 4) for _ in range(rng.randint(1, 3))]
    assert weeks_circular(m) == weeks_circular_brute(m), (m, weeks_circular(m), weeks_circular_brute(m))
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", weeks_circular([5, 2, 1]), weeks_circular([1, 2, 3]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 6 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def numberOfWeeks(milestones):
    total, mx = sum(milestones), max(milestones)
    rest = total - mx
    return total if mx <= rest + 1 else 2 * rest + 1
    # tương đương: rest + min(mx, rest + 1)
```

**Checklist:**
- [ ] So `mx` với **tổng các dự án khác** (không phải dự án lớn thứ hai)?
- [ ] Nhánh lấn át trả `2·rest + 1` (có `+1`)?
- [ ] `[9] → 1`, `[1] → 1`?
- [ ] Không mô phỏng từng tuần (`S` tới `10¹⁴`)?
- [ ] `O(n)`.